# A full business solution
## Business Challange:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits . 
We will be provided with company name and their primary website.

In [5]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import WebsiteScraper


In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key) >10:
    print("API key is valid")
else:
    print("API key is invalid")
MODEL = "gpt-5-nano"
openai = OpenAI()

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be the most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    """
    scraper = WebsiteScraper(url)
    links = scraper.get_links()
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.flightaware.com/live/"))

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)  }
        ],
        response_format = {"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://www.flightaware.com/live/")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def get_page_content(url):
    return WebsiteScraper(url).get_content()

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = get_page_content(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links: \n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += get_page_content(url)
    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://www.flightaware.com/live/"))

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("FlightAware", "https://www.flightaware.com/live/")

In [18]:
from IPython.display import Markdown


def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("FlightAware", "https://www.flightaware.com/live/")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI, **but this time in German**,
with the familiar typewriter animation

In [20]:
from IPython.display import update_display

brochure_system_prompt_german = """
You are a marketing assistant.

Generate the brochure entirely in German.
Use natural, professional German.
Keep headings, bullet points, and formatting in German.
"""

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt_german},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("FlightAware", "https://www.flightaware.com/live/")